In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)

zip_path = "/content/drive/MyDrive/tomato.zip"
extract_path = "/content/tomato"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset berhasil diekstrak")

TRAIN_DIR = "/content/tomato/tomato/train"
VAL_DIR = "/content/tomato/tomato/val"

IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names

MODEL_PATH = "/content/drive/MyDrive/final_resnet50_2.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model berhasil dimuat")

y_true = []
y_pred = []

for images, labels in val_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        np.argmax(
            predictions,
            axis=1
        )
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average='macro'
)

bal_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

print("\n" + "="*50)
print("RINGKASAN METRIK UNTUK BAB 4")
print("="*50)

print(f"Accuracy          : {acc:.4f} ({acc*100:.2f}%)")
print(f"Macro F1-Score    : {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"Balanced Accuracy : {bal_acc:.4f} ({bal_acc*100:.2f}%)")

print("="*50)

metrics_df = pd.DataFrame({
    "Metric":[
        "Accuracy",
        "Macro F1-Score",
        "Balanced Accuracy"
    ],
    "Value":[
        acc,
        macro_f1,
        bal_acc
    ]
})

metrics_df.to_csv(
    "/content/drive/MyDrive/final_resnet50_2_bab4_metrics.csv",
    index=False
)

report_text = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)

print("\n")
print(report_text)

with open(
    "/content/drive/MyDrive/final_resnet50_2_classification_report.txt",
    "w"
) as f:
    f.write(report_text)

report_df = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True
    )
).transpose()

report_df.to_csv(
    "/content/drive/MyDrive/final_resnet50_2_classification_report.csv"
)

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    "Confusion Matrix ResNet50"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/final_resnet50_2_confusion_matrix.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("\nSemua file berhasil disimpan ke Google Drive")